In [ ]:
import torch
from torchvison.models import resnet50, ResNet50_Weights
import torchvision.transforms.functional as transform
import PIL
import os
from tqdm import tqdm
import json
import numpy as np

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

model = (скачиваем .pth)

In [ ]:
device = torch.device("cpu")

In [ ]:
# смотрим, какой слой у сети последний - дб линейный на 31 класс
model.to(device)

In [ ]:
# у него дал самый последний слой на 100 классов
model.fc

In [ ]:
# тождественное преобразование
class Identify(torch.nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return x

In [ ]:
model.fc = Identify()

In [ ]:
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

In [ ]:
PIL_img = PIL.Image.open(img_path)
tensor_img = transform.to_tensor(PIL_img)
new_PIL_img = transform.to_pil_image(tensor_img)

In [ ]:
pil_img = PIL.Image.open("/exp.png")

In [ ]:
pil_img

In [ ]:
img = transform.to_tensor(pil_img)

In [ ]:
img.shape # 3.224.224

In [ ]:
img_transformed = preprocess(img)
img_transformed.shape

In [ ]:
# модель -  врежим вычисления, а не обучения
model.eval()

In [ ]:
# картинку - в батч
batch_img = preprocess(img).unsqueeze(0)
batch_img.shape # 1.3.224.224

In [ ]:
with torch.no_grad():
  prediction = model(batch_img)

prediction.shape # [1, N] -массив

In [ ]:
files = os.listdir('./pics')
len(files)

In [ ]:
imgs = []

for f in tqdm(files):
  pil_img = PIL.Image.open(f'./pics/{f}')
  img = transform.to_tensor(pil_img)
  img_transformed = preprocess(img)
  imgs.append(img_transformed)

In [ ]:
len(imgs)

In [ ]:
index = []
for f in files:
  index.append(f.split('_')[0])

with open('index.json', 'w') as f:
  json.dump(index, f)

In [ ]:
imgs(0).shape #3.224.224

In [ ]:
batch = torch.vstack(tuple((im.unsqeeze(0) for im in imgs)))

bach.shape # N.3.224.224

In [ ]:
%%time

with torch.no_grad():
  vects = model(batch)

vects.shape # torch.size(N, 2048)

In [ ]:
vects_norm = vects / torch.Tensor.repeat(vects.norm(dim=1).unsqeeze(1), 1, 2048)
vects_norm_np = vects_norm.numpy()

In [ ]:
np.min(vects_norm_np)

In [ ]:
np.max(vects_norm_np)

In [ ]:
with open('./vects.npy', 'wb') as f:
  np.save(f, vects_norm_np)

In [ ]:
vects_norm_np = vects_norm_np.astype(np.float16)
vects_norm_np_q = vects_norm_np * 128
vects_norm_np_q = vects_norm_np_q.astype(np.int8)

with open('./vects_q.npy', 'wb') as f:
  np.save(f, vects_norm_np_q)

## дальше - рекомендашки

In [ ]:
with open('./vects_q.npy', 'rb') as f:
  vects = np.load(f)

with open('./index.json', 'rb') as f:
  index = json.load(f)

In [ ]:
def get_similar(v, vects, n=10):
  scores = np.matmul(vects, v)
  scores = scores / 128
  top_similat_ind = (-scores).argsort()[:n]
  return {
      'similar_ind': list(top_similat_ind),
      'similar_scores': list(scores[top_similat_ind])
  }

In [ ]:
def get_by_indexs(inds, index):
  f_names = []
  for i in inds:
    f_names.append(f'/pics/{index[i]}_0.jpg')
  return f_names

In [ ]:
def get_sim_mean(viewed_ids, vects, index, n=10):
  v = np.zeros(2048)
  viewed_ids = viewed_ids[::-1][:10]
  for i in viewed_ids[:]:
    v += vects[i]
  v /= len(viewed_ids)
  v / = np.linalg.norm(v)
  return filter_(get_similar(v, vects, n+len(viewed_ids)), viewed_ids)


In [ ]:
def filter_(recs, viewed_ids):
  res = {
      'similar_ind': [],
      'similar_scores': []
  }
  for i in range(len(recs['similar_ind'])):
    if recs[similar_ind][i] not in viewed_ids:
      res['similar_ind'].append(recs[similar_ind][i])
      res['similar_scores'].append(recs[similar_scores][i])
  return res

In [ ]:
class NpEncoder(json.JSONEncoder):
  def default(self, obj):
    if isinstance(obj, np.integer):
      return int(obj)
    if isinstance(obj, np.floating):
      return float(obj)
    if isinstance(obj, np.ndarray):
      return jbj.tolist()
    return super(NpEncoder, self).default(obj)


In [ ]:
viewed_ids = [1]

In [ ]:
res = get_sim_mean(viewed_ids, vects)

In [ ]:
res['similar_ind']

In [ ]:
json.dumps(res, cls=NpEncoder)